In [ ]:
import snowflake.connector
import polars as pl
import pandas as pd
from IPython.display import display
import gc


# Configure Polars display settings
pl.Config.set_tbl_cols(-1)  # Limit to 10 columns
pl.Config.set_tbl_rows(-1)  # Limit to 20 rows
pl.Config.set_tbl_width_chars(200)  # Set the width of columns to 25 characters
pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_hide_dtype_separator(True)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_cell_alignment("RIGHT")
pl.Config.float_precision=3

gc.collect()

In [ ]:

def get_connection():
    return snowflake.connector.connect(
         account = 'tata.us-east-1',
        user = 'user',
        password = 'pwd',
        database='RAW',
        schema='MARKETO',
        warehouse='BSM_MQA_AMPLIFY_ABSM',
        role='BSM_DEVELOPER',
        paramstyle="qmark",
        client_session_keep_alive=True,  # Reuse sessions
        network_timeout=30  # Prevent hanging connections
    )


def execute_sql(sql: str) -> pl.DataFrame:
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute(sql)
        # Column names
        column_names = [col[0] for col in cur.description]
        result = cur.fetchall()  # Changed from fetchmany(100) to fetchall()
        if result:
            return pl.DataFrame(result, schema=column_names)
        else:
            return pl.DataFrame(schema=column_names)
    finally:
        cur.close()  # Added cursor close
        conn.close()   



In [ ]:
# _2025_data_path = r"C:\Users\mandald\Downloads\FACEBOOK_2025.xlsx"
# _2024_data_path = r"C:\Users\mandald\Downloads\Facebook_2024.xlsx"

# _2025_data = pl.read_excel(_2025_data_path).select(pl.lit('2025').alias('year_raw'), pl.col('Account ID').cast(pl.Utf8).alias('account_id_raw').unique() )
# _2024_data = pl.read_excel(_2024_data_path).select(pl.lit('2024').alias('year_raw'), pl.col('Account ID').cast(pl.Utf8).alias('account_id_raw').unique() )
# _mpm_data = execute_sql(r"select distinct year(date)::string as year, account_id::string as account_id from RAW.FACEBOOK.FACEBOOK_STANDARD_RAW where year(date) in (2024,2025) order by 1,2")

# raw_data_union = pl.concat([_2024_data, _2025_data], how="vertical")

# _mpm_data = _mpm_data.rename({
#     "YEAR": "year",
#     "ACCOUNT_ID": "account_id"
# })
# # display(raw_data_union.count())
# # display(_mpm_data.count())

# raw_mpm_join_data = raw_data_union.join(
#     _mpm_data.with_columns(
#         pl.col("year").alias("year_mpm"),
#         pl.col("account_id").alias("account_id_mpm")
#     ),
#     left_on=["year_raw", "account_id_raw"],
#     right_on=["year_mpm", "account_id_mpm"],
#     how="left"
# )


# display(raw_mpm_join_data)

### Facebook

In [ ]:
fb_query = r"""
WITH accounts AS (
    SELECT column1 AS account_id
    FROM VALUES
        ('1250054443349213'),
        ('1288226306007697'),
        ('173852499053980'),
        ('383920684018076'),
        ('592442635932034'),
        ('389231205102691')
)

SELECT 
    a.account_id,
    COALESCE(SUM(f.clicks), 0)              AS sum_clicks,
    COALESCE(SUM(f.impressions), 0)         AS sum_impressions,
    COALESCE(SUM(f.cost_per_unique_click), 0) AS sum_cost
FROM accounts a
LEFT JOIN RAW.FACEBOOK.FACEBOOK_STANDARD_RAW f
    ON a.account_id = f.account_id
GROUP BY a.account_id
ORDER BY a.account_id;
"""

display(execute_sql(fb_query))

In [ ]:
linkedin_query= r"""
WITH accounts AS (
    SELECT column1 AS account_id
    FROM VALUES
        ('505596333'), ('512767775'), ('512376482'), ('513980103'),
        ('503845942'), ('515021027'), ('514217705'), ('509489795'),
        ('514845605'), ('514977720'), ('510577257'), ('515020045'),
        ('514865104'), ('516749427'), ('513904283'), ('516723050'),
        ('508619077'), ('515419084'), ('516417355'), ('503898391'),
        ('515421590'), ('514980106'), ('506904624'), ('517341024')
)

SELECT
    a.account_id,
    COALESCE(SUM(b.clicks), 0)              AS sum_clicks,
    COALESCE(SUM(b.impressions), 0)         AS sum_impressions,
    COALESCE(SUM(b.cost_in_usd), 0) AS sum_cost
FROM accounts a
LEFT JOIN RAW.LINKEDIN_ADS.CAMPAIGN_HISTORY ch
    ON a.account_id = ch.account_id
LEFT JOIN RAW.LINKEDIN_ADS.AD_ANALYTICS_BY_CAMPAIGN b
    ON ch.id = b.campaign_id
GROUP BY a.account_id
ORDER BY a.account_id
;
"""

display(execute_sql(linkedin_query))